# TabPFNによる二値分類

[公式サンプル](https://github.com/PriorLabs/TabPFN/blob/main/examples/tabpfn_for_binary_classification.py)と同じ乳がん診断データを使い、LogisticRegressionと比較する。初回の`fit`時にはモデルのダウンロードとライセンス同意を求められる場合がある。

In [1]:
!pip install -q -U tabpfn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.2/753.2 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.9/370.9 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tabpfn import TabPFNClassifier

from tabpfn.constants import ModelVersion

In [3]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)
print(X_train.shape, X_test.shape)

(381, 30) (188, 30)


In [4]:
#tabpfn = TabPFNClassifier()
tabpfn = TabPFNClassifier.create_default_for_version(ModelVersion.V2)

tabpfn.fit(X_train, y_train)

logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logistic.fit(X_train, y_train)

tabpfn-v2-classifier-finetuned-zk73skhh.(…):   0%|          | 0.00/29.0M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression(max_iter=1000))])

In [5]:
for name, model in {"TabPFN": tabpfn, "LogisticRegression": logistic}.items():
    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = model.predict(X_test)
    print(name)
    print("  ROC AUC:", roc_auc_score(y_test, probabilities))
    print("  Accuracy:", accuracy_score(y_test, predictions))

TabPFN
  ROC AUC: 0.9970395954113729
  Accuracy: 0.973404255319149
LogisticRegression
  ROC AUC: 0.9972862957937585
  Accuracy: 0.9787234042553191
